# Preprocessing of 2023 SKT Living Population Data by Hour

This code aggregates and preprocesses restricted-access SKT living population data into a usable format for flood vulnerability assessment.

To ensure consistency with other socioeconomic vulnerability datasets, only records from 2023 were selected. The data were aggregated by administrative dong and time of day to examine differences in population distribution between daytime and nighttime. The processed data were used to assess daytime and nighttime population exposure to potential flood hazards.

The preprocessing was conducted within the Busan Big Data Center's secure environment.

In [ ]:
import pandas as pd
import os

# Setting
pop011_folder = "D:/20260914_권나영/POP008" # P011 data are stored in the same directory as P008 data.
start_period = 202301
end_period = 202312

# Read the csv file
df = pd.read_csv(os.path.join(pop011_folder, "POP011_202301.csv"), sep="|")
rows, columns = df.shape
print(f"데이터프레임의 행 수: {rows}") # Number of rows in the DataFrame
print(f"데이터프레임의 열 수: {columns}") # Number of columnss in the DataFrame

columns_list = df.columns.tolist()
print("데이터프레임의 컬럼 이름 리스트:", columns_list) # List of column names in the DataFrame

데이터프레임의 행 수: 5431414
데이터프레임의 열 수: 75
데이터프레임의 컬럼 이름 리스트: ['기준일자', '행정동코드', '격자id', '00시_주거', '01시_주거', '02시_주거', '03시_주거', '04시_주거', '05시_주거', '06시_주거', '07시_주거', '08시_주거', '09시_주거', '10시_주거', '11시_주거', '12시_주거', '13시_주거', '14시_주거', '15시_주거', '16시_주거', '17시_주거', '18시_주거', '19시_주거', '20시_주거', '21시_주거', '22시_주거', '23시_주거', '00시_직장', '01시_직장', '02시_직장', '03시_직장', '04시_직장', '05시_직장', '06시_직장', '07시_직장', '08시_직장', '09시_직장', '10시_직장', '11시_직장', '12시_직장', '13시_직장', '14시_직장', '15시_직장', '16시_직장', '17시_직장', '18시_직장', '19시_직장', '20시_직장', '21시_직장', '22시_직장', '23시_직장', '00시_방문', '01시_방문', '02시_방문', '03시_방문', '04시_방문', '05시_방문', '06시_방문', '07시_방문', '08시_방문', '09시_방문', '10시_방문', '11시_방문', '12시_방문', '13시_방문', '14시_방문', '15시_방문', '16시_방문', '17시_방문', '18시_방문', '19시_방문', '20시_방문', '21시_방문', '22시_방문', '23시_방문']


In [4]:
df.head()

,기준일자,행정동코드,격자id,00시_주거,01시_주거,02시_주거,03시_주거,04시_주거,05시_주거,06시_주거,...,14시_방문,15시_방문,16시_방문,17시_방문,18시_방문,19시_방문,20시_방문,21시_방문,22시_방문,23시_방문
0,20230101,26110510,197029,10.06,10.64,10.83,10.98,11.15,11.19,11.05,...,39.37,42.92,43.42,40.23,33.30,27.50,17.16,13.22,10.85,8.21
1,20230101,26110510,197028,15.61,16.52,16.80,17.04,17.30,17.36,17.14,...,61.09,66.59,67.38,62.43,51.67,42.66,26.63,20.51,16.83,12.73
2,20230101,26110510,198264,19.73,20.87,21.23,21.53,21.87,21.94,21.67,...,77.20,84.16,85.15,78.90,65.29,53.92,33.65,25.93,21.27,16.09
3,20230101,26110510,197027,27.07,28.65,29.13,29.55,30.01,30.12,29.73,...,105.95,115.50,116.85,108.27,89.61,73.99,46.18,35.58,29.19,22.08
4,20230101,26110510,198263,11.19,11.84,12.04,12.21,12.40,12.44,12.29,...,43.77,47.72,48.28,44.74,37.02,30.57,19.08,14.70,12.06,9.12


### Inspect the original data

In [ ]:
# Check the temporal and spatial data structure
print("Number of unique dates:", df["기준일자"].nunique())
print("First date:", df["기준일자"].min())
print("Last date:", df["기준일자"].max())
print("Number of unique grid cells:", df["격자id"].nunique())
print("Number of administrative dongs:", df["행정동코드"].nunique())

# Check for duplicate date-grid combinations
print("Duplicate date-grid records:", df.duplicated(["기준일자", "격자id"]).sum())

# Check for missing values across the entire DataFrame
print("Total missing values:", df.isna().sum().sum())

# Check for negative values in population columns
pop_cols = df.columns[3:]
print("Number of negative values:", (df[pop_cols] < 0).sum().sum())


날짜 수: 31
첫 날짜: 20230101
마지막 날짜: 20230131
격자 수: 176021
행정동 수: 206
날짜-격자 중복: 0
전체 결측치: 0
음수값 개수: 0


### Check Hourly Population by Activity Type (Home, Work, and Visit)

In [ ]:
gid = df["격자id"].iloc[0]

# Check hourly values for the selected grid cell on 2023-01-01
sample = df[(df["기준일자"]==20230101)&(df["격자id"]==gid)]

# Inspect hourly population by activity type for a selected grid cell
for h in range(24):
    hh = f"{h:02d}시"
    
    home = sample[f"{hh}_주거"].iloc[0]
    work = sample[f"{hh}_직장"].iloc[0]
    visit = sample[f"{hh}_방문"].iloc[0]
    
    print(hh, "주거:",home, "직장:",work, "방문:",visit, "합계:", home+work+visit)

00시 주거: 10.06 직장: 1.59 방문: 17.96 합계: 29.61
01시 주거: 10.64 직장: 1.08 방문: 15.72 합계: 27.44
02시 주거: 10.83 직장: 1.01 방문: 14.45 합계: 26.29
03시 주거: 10.98 직장: 0.93 방문: 13.35 합계: 25.259999999999998
04시 주거: 11.15 직장: 0.88 방문: 12.66 합계: 24.69
05시 주거: 11.19 직장: 0.97 방문: 12.59 합계: 24.75
06시 주거: 11.05 직장: 1.45 방문: 13.52 합계: 26.02
07시 주거: 10.49 직장: 2.51 방문: 12.45 합계: 25.45
08시 주거: 10.21 직장: 3.9 방문: 13.02 합계: 27.130000000000003
09시 주거: 10.08 직장: 6.81 방문: 15.2 합계: 32.09
10시 주거: 9.87 직장: 12.03 방문: 17.5 합계: 39.4
11시 주거: 9.4 직장: 13.37 방문: 20.22 합계: 42.989999999999995
12시 주거: 8.98 직장: 16.45 방문: 21.78 합계: 47.21
13시 주거: 8.7 직장: 16.52 방문: 32.47 합계: 57.69
14시 주거: 8.51 직장: 16.98 방문: 39.37 합계: 64.86
15시 주거: 8.45 직장: 17.34 방문: 42.92 합계: 68.71000000000001
16시 주거: 8.56 직장: 13.88 방문: 43.42 합계: 65.86
17시 주거: 8.79 직장: 12.9 방문: 40.23 합계: 61.919999999999995
18시 주거: 9.1 직장: 11.24 방문: 33.3 합계: 53.64
19시 주거: 9.51 직장: 9.15 방문: 27.5 합계: 46.16
20시 주거: 10.12 직장: 7.38 방문: 17.16 합계: 34.66
21시 주거: 10.89 직장: 3.23 방문: 13.22 합계: 27.3400

In [ ]:
# Calculate hourly total population and daily average population
hours = [f"{h:02d}시" for h in range(24)]

for h in hours:
    df[h] = df[h+"_주거"]+df[h+"_직장"]+df[h+"_방문"]
    
df["전체"] = df[hours].mean(axis=1)
df[["기준일자", "격자id", "전체"]].head()

,기준일자,격자id,전체
0,20230101,197029,38.536250
1,20230101,197028,59.791250
2,20230101,198264,75.566250
3,20230101,197027,103.696250
4,20230101,198263,42.845417


In [ ]:
# Print hourly total population
for h in hours:
    print(h, df[h].mean())

00시 20.531505705548213
01시 20.499181482060763
02시 20.474091288238075
03시 20.438888296903475
04시 20.39234629881312
05시 20.343617091279327
06시 20.291221425983444
07시 20.278463991173492
08시 20.44726923598939
09시 20.61886724528364
10시 20.624543675021613
11시 20.632158793288564
12시 20.637632942367574
13시 20.646423491205002
14시 20.62589522728899
15시 20.571924793438274
16시 20.601877940088205
17시 20.627114095897678
18시 20.631532750045906
19시 20.625893049238996
20시 20.63111156876585
21시 20.64284447478972
22시 20.648934785708523
23시 20.659201060746412


### Calculate average population for daytime and nighttime

In [16]:
day = [f"{h:02d}시" for h in range(7, 19)]
night = [f"{h:02d}시" for h in list(range(19, 24))+list(range(7))]

df["주간"] = df[day].mean(axis=1)
df["야간"] = df[night].mean(axis=1)

df[["기준일자", "격자id", "전체", "주간", "야간"]].head()

,기준일자,격자id,전체,주간,야간
0,20230101,197029,38.536250,48.912500,28.160000
1,20230101,197028,59.791250,75.893333,43.689167
2,20230101,198264,75.566250,95.915000,55.217500
3,20230101,197027,103.696250,131.620833,75.771667
4,20230101,198263,42.845417,54.382500,31.308333


### Calculate monthly average population by grid cell (Total, Daytime, and Nighttime)

In [ ]:
import pandas as pd
import os

pop011_folder = "D:/20260914_권나영/POP008" # Same as P008

result = []
for m in range(1, 13):
    file = f"POP011_2023{m:02d}.csv"
    x = pd.read_csv(os.path.join(pop011_folder, file), sep="|")
    x_columns = x.columns.str.replace('"', '', regex=False)

    day_h = list(range(7, 19))
    night_h = list(range(19,24))+list(range(0,7))

    all_cols = [f"{h:02d}시_{t}" for h in range(24) for t in ["주거", "직장", "방문"]]
    day_cols = [f"{h:02d}시_{t}" for h in day_h for t in ["주거", "직장", "방문"]]
    night_cols = [f"{h:02d}시_{t}" for h in night_h for t in ["주거", "직장", "방문"]]

    # Calculate average hourly population
    x["전체"] = x[all_cols].sum(axis=1)/24
    x["주간"] = x[day_cols].sum(axis=1)/12
    x["야간"] = x[night_cols].sum(axis=1)/12

    temp = x.groupby(["행정동코드", "격자id"])[["전체", "주간", "야간"]].agg(["sum", "count"])

    result.append(temp)

    print(m, "월 완료")

1 월 완료
2 월 완료
3 월 완료
4 월 완료
5 월 완료
6 월 완료
7 월 완료
8 월 완료
9 월 완료
10 월 완료
11 월 완료
12 월 완료


### Calculate annual average population by grid cell for 2023

In [ ]:
# Combine monthly aggregated data from all 12 months
total = pd.concat(result).groupby(level=[0,1]).sum()

# Calculate annual average population
final = pd.DataFrame(index=total.index)

final["전체"] = total[("전체", "sum")]/total[("전체", "count")]
final["주간"] = total[("주간", "sum")]/total[("주간", "count")]
final["야간"] = total[("야간", "sum")]/total[("야간", "count")]

final = final.reset_index()
print(final.shape)
#final.head()

final[["전체", "주간", "야간"]].describe()

(186433, 5)


,전체,주간,야간
count,186433.000000,186433.000000,186433.000000
mean,19.035576,18.902482,19.168670
std,47.809836,47.515199,48.645663
min,0.000417,0.000000,0.000000
25%,0.190255,0.207525,0.166671
50%,1.625328,1.757320,1.467658
75%,16.284229,16.274027,16.102575
max,1553.695201,1322.924749,1784.465653


### Save

In [10]:
final.to_csv(r"D:/20260914_권나영/POP011_2023_전체_주간_야간_연평균.csv", index=False, encoding="utf-8-sig")